# HOI Pose Estimation - Main Notebook

**PoliTo Advanced ML 2026-27** - Hand-Object Interaction with Transformers (HaMeR) on DexYCB

This single notebook runs the complete workflow:
1. **Setup** - Install deps, mount Drive, download dataset
2. **Data Exploration** - Inspect DexYCB, verify active object logic
3. **HaMeR Inference** - Pretrained evaluation (Table 1)
4. **HaMeR Fine-tuning** - Train on DexYCB
5. **HOI Training** - Full hand-object model
6. **Evaluation** - Table 2 metrics + visualizations

**Run: Runtime → Run all**

In [ ]:
# ============================================================
# 1. SETUP - Install uv, clone repo, install deps, mount Drive
# ============================================================

# Install uv
!pip install uv -q
print("uv installed")

# Clone repository
import os

REPO_URL = "https://github.com/mikiungolo/Estimating-Poses-of-Hands-Interacting-with-Objects.git"
REPO_PATH = "/content/Estimating-Poses-of-Hands-Interacting-with-Objects"

if not os.path.exists(REPO_PATH):
    print(f"Cloning from {REPO_URL}...")
    !git clone {REPO_URL} {REPO_PATH}
else:
    print("Repository exists, pulling updates...")
    %cd {REPO_PATH}
    !git pull

%cd {REPO_PATH}
print(f"Repository at {REPO_PATH}")

In [ ]:
# Install dependencies with uv
!uv sync -q
print("Dependencies installed")

In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
print("Google Drive mounted")

In [ ]:
# Create persistent folders on Drive
import os

DRIVE_PROJECT = "/content/drive/MyDrive/HOI_Project"

dirs = [
    f"{DRIVE_PROJECT}/checkpoints/hamer_pretrained",
    f"{DRIVE_PROJECT}/checkpoints/hamer_finetuned",
    f"{DRIVE_PROJECT}/checkpoints/hoi_model",
    f"{DRIVE_PROJECT}/dataset/raw",
    f"{DRIVE_PROJECT}/dataset/processed",
    f"{DRIVE_PROJECT}/logs",
    f"{DRIVE_PROJECT}/outputs",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

# Symlink for easy access
for src, dst in [
    (f"{DRIVE_PROJECT}/checkpoints", "/content/checkpoints"),
    (f"{DRIVE_PROJECT}/dataset", "/content/dataset"),
    (f"{DRIVE_PROJECT}/logs", "/content/logs"),
]:
    if not os.path.islink(dst):
        os.symlink(src, dst)

print(f"Drive structure created at {DRIVE_PROJECT}")
print("  Checkpoints: /content/checkpoints -> Drive")
print("  Dataset:     /content/dataset -> Drive")
print("  Logs:        /content/logs -> Drive")

In [ ]:
# Download dataset DexYCB subset
!python scripts/download_dataset.py --output /content/dataset/raw
print("Dataset downloaded")

In [ ]:
# Verify imports
import sys

sys.path.insert(0, REPO_PATH)

import torch
import wandb

from src.data import DexYCBDataset, get_transforms
from src.utils.checkpoint import load_checkpoint
from src.utils.config import load_config
from src.utils.metrics import compute_mpjpe

config = load_config("configs/config.yaml")
print(f"Config loaded: batch_size={config.data.batch_size}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"WandB: {wandb.__version__}")

print("\n✅ SETUP COMPLETE - Ready for Data Exploration")


# ============================================================
# 2. DATA EXPLORATION - Inspect DexYCB dataset
# ============================================================

In [ ]:
# Create dataset and dataloader
transform = get_transforms("train", config)
train_dataset = DexYCBDataset("train", config.data, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=config.data.batch_size, num_workers=0, shuffle=True)

print(f"Train dataset size: {len(train_dataset)}")

In [ ]:
# Inspect a batch
import numpy as np

batch = next(iter(train_loader))

print("Batch keys:", batch.keys())
for k, v in batch.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    elif hasattr(v, 'shape'):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v)}")

In [ ]:
# Visualize RGB images from batch
import matplotlib.pyplot as plt

rgb_batch = batch['rgb']  # (B, 3, H, W)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(min(8, rgb_batch.shape[0])):
    ax = axes[i // 4, i % 4]
    img = rgb_batch[i].permute(1, 2, 0).numpy()  # (H, W, 3)
    img = (img * 255).astype(np.uint8)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"Sample {i}")
plt.tight_layout()
plt.show()


# ============================================================
# 3. HAME PRETRAINED INFERENCE - Table 1 metrics
# ============================================================

In [ ]:
# Load pretrained HaMeR
import torch

from src.models.hamer import HaMeR

device = torch.device(config.device)
model = HaMeR.load_from_checkpoint(config.hamer.pretrained_path).to(device)
model.eval()

print(f"HaMeR pretrained loaded on {device}")

In [ ]:
# Validation dataloader
val_transform = get_transforms("val", config)
val_dataset = DexYCBDataset("val", config.data, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=config.data.batch_size, num_workers=0)

print(f"Val dataset size: {len(val_dataset)}")

In [ ]:
# Evaluation loop (requires MANO layer for vertex metrics)

all_mpjpe = []
all_mpvpe = []

with torch.no_grad():
    for batch in val_loader:
        rgb = batch['rgb'].to(device)
        pred = model(rgb)

        # Joint metrics
        if 'joints_3d' in batch:
            mpjpe = compute_mpjpe(pred['joints_3d'], batch['joints_3d'].to(device))
            all_mpjpe.append(mpjpe)

        # Vertex metrics require MANO layer (TODO)
        # if mano_layer:
        #     mpvpe = compute_mpvpe(...)
        #     all_mpvpe.append(mpvpe)

if all_mpjpe:
    print(f"MPJPE: {np.mean(all_mpjpe):.2f} mm")
print("\nTODO: Implement full Table 1 metrics with MANO layer (smplx)")


# ============================================================
# 4. HAME FINE-TUNING on DexYCB
# ============================================================

In [ ]:
# Model, optimizer, loss
import torch

from src.models.hamer import HaMeR
from src.training import Trainer

model = HaMeR(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(),
    lr=config.train.hamer.lr,
    weight_decay=config.train.hamer.weight_decay)
# loss_fn = ... (implement HaMeR loss in src/models/losses.py)

# Dataloaders
train_transform = get_transforms("train", config)
val_transform = get_transforms("val", config)

train_dataset = DexYCBDataset("train", config.data, transform=train_transform)
val_dataset = DexYCBDataset("val", config.data, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=config.data.batch_size,
                          num_workers=0, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.data.batch_size,
                        num_workers=0)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Training (implement Trainer in src/training/trainer.py)
# trainer = Trainer(model, train_loader, val_loader, optimizer, loss_fn,
#                   config, device, compute_metrics_fn)
# trainer.fit(config.train.hamer.epochs, run_name="hamer-finetuned")

print("TODO: Implement HaMeR loss and training loop")
print(f"Epochs: {config.train.hamer.epochs}, LR: {config.train.hamer.lr}")


# ============================================================
# 5. HOI MODEL TRAINING - Full hand-object interaction
# ============================================================

In [ ]:
# Model, optimizer, loss
from src.models.hoi import HOIModel
from src.models.losses import HOILoss

model = HOIModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(),
    lr=config.train.hoi.lr,
    weight_decay=config.train.hoi.weight_decay)
loss_fn = HOILoss(config)

# Dataloaders
train_transform = get_transforms("train", config)
val_transform = get_transforms("val", config)

train_dataset = DexYCBDataset("train", config.data, transform=train_transform)
val_dataset = DexYCBDataset("val", config.data, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=config.data.batch_size,
                          num_workers=0, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.data.batch_size,
                        num_workers=0)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Training
trainer = Trainer(model, train_loader, val_loader, optimizer, loss_fn,
                  config, device)
trainer.fit(config.train.hoi.epochs, run_name="hoi-training")

print("HOI training complete")


# ============================================================
# 6. EVALUATION - Table 2 metrics + Visualizations (Figure 4)
# ============================================================

In [ ]:
# Load trained HOI model
from src.models.hoi import HOIModel

model = HOIModel(config).to(device)
checkpoint_path = f"{config.checkpoint_dir}/hoi_model/best.pth"
epoch, metrics = load_checkpoint(checkpoint_path, model, map_location=device)
model.eval()

print(f"Loaded checkpoint from epoch {epoch}")

In [ ]:
# Test dataloader
test_transform = get_transforms("test", config)
test_dataset = DexYCBDataset("test", config.data, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=config.data.batch_size, num_workers=0)

print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
# Evaluation loop - Table 2 metrics
from src.utils.metrics import compute_mpjpe

all_metrics = []

with torch.no_grad():
    for batch in test_loader:
        rgb = batch['rgb'].to(device)
        obj_class = batch['obj_class'].to(device)
        pred = model(rgb, obj_class)

        # Compute metrics (requires MANO layer for vertex metrics)
        # metrics = compute_all_metrics(pred, batch, mano_layer, obj_mesh)
        # all_metrics.append(metrics)
        pass

print("TODO: Implement full Table 2 metrics with MANO layer (smplx)")
print("Target metrics: MPJPE, PA-MPJPE, MPVPE, PA-MPVPE, AUC, RR, ADD, PD")

In [ ]:
# Qualitative visualizations (Figure 4)

# for batch in test_loader:
#     rgb = batch['rgb'].to(device)
#     obj_class = batch['obj_class'].to(device)
#     pred = model(rgb, obj_class)
#
#     # Visualize predictions
#     # vis = overlay_predictions(rgb[0].cpu().numpy(), pred, ...)
#     pass

print("TODO: Implement qualitative visualizations (Figure 4)")
print("Requires: pyrender, trimesh, MANO layer, YCB object meshes")


---
## Keep-Alive (Optional)

To prevent disconnect after 12h inactivity, open browser console (F12) and paste:

```javascript
setInterval(() => {
  const btn = document.querySelector('#connect-button');
  if (btn) btn.click();
  console.log('Keep-alive ping');
}, 5 * 60 * 1000);
```

Or install Chrome extension "Colab Keep Alive".